In [ ]:
pip install transformers torch accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Indonesia

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/mental_health/reddit_indonesia_raw.csv")

In [ ]:
def llm_filter(text):

    prompt = f"""
Analyze the following Reddit post from an Indonesian cultural context.
Think step-by-step:
1. Identify distress: List the emotional states (e.g., sadness, feeling trapped, loneliness).
2. Identify cultural signals: List cultural elements (e.g., strict parenting, filial expectations, family conflict, socioeconomic pressure in Indonesia).
3. Final Decision: Based on the above, is this a personal distress message?
   (Must be a first-person account of an individual's struggle).

Format:
THOUGHTS: [Brief reasoning]
DISTRESS: [YES/NO]
CULTURE: [YES/NO]

Text: "{text}"
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=50
    )

    result = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    distress = "distress: yes" in result
    culture = "culture: yes" in result

    return distress and culture

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
filtered_rows = []

batch_size = 10

for i in range(0, len(df), batch_size):

    batch = df.iloc[i:i+batch_size]

    print(f"Processing batch {i} - {min(i+batch_size, len(df))}")

    for _, row in batch.iterrows():

        text = row["selftext"]

        try:
            if not isinstance(text, str) or len(text) < 20:
                continue

            if llm_filter(text):
                filtered_rows.append(row)

        except Exception as e:
            print("error:", e)

    if i % 5 == 0:
        temp_df = pd.DataFrame(filtered_rows)
        temp_df.to_csv("/content/drive/MyDrive/mental_health/backup_filtered_indonesia.csv", index=False)


df_filtered = pd.DataFrame(filtered_rows).reset_index(drop=True)

df_filtered.to_csv(
    "/content/drive/MyDrive/mental_health/filtered_llm_indonesia.csv",
    index=False
)

Processing batch 0 - 10
Processing batch 10 - 20
Processing batch 20 - 30
Processing batch 30 - 40
Processing batch 40 - 50
Processing batch 50 - 60
Processing batch 60 - 70
Processing batch 70 - 80
Processing batch 80 - 90
Processing batch 90 - 100
Processing batch 100 - 110
Processing batch 110 - 120
Processing batch 120 - 130
Processing batch 130 - 140
Processing batch 140 - 150
Processing batch 150 - 160
Processing batch 160 - 170
Processing batch 170 - 180
Processing batch 180 - 190
Processing batch 190 - 200
Processing batch 200 - 210
Processing batch 210 - 220
Processing batch 220 - 230
Processing batch 230 - 240
Processing batch 240 - 250
Processing batch 250 - 260
Processing batch 260 - 270
Processing batch 270 - 280
Processing batch 280 - 290
Processing batch 290 - 300
Processing batch 300 - 310
Processing batch 310 - 320
Processing batch 320 - 330
Processing batch 330 - 340
Processing batch 340 - 350
Processing batch 350 - 360
Processing batch 360 - 370
Processing batch 370 

## Spanish

In [ ]:
def llm_filter_spanish(text):

    prompt = f"""
Analyze the following Reddit post from an Spanish-Speaking cultural context.
Think step-by-step:
1. Identify distress: List the emotional states (e.g., sadness, feeling trapped, loneliness).
2. Identify cultural signals: List cultural elements (e.g.Religion and moral expectations, Strong family obligation, Respect for hierarchy, Social image and reputation, work and economic pressure in Latin America/Spain, Humor as a coping mechanism).
3. Final Decision: Based on the above, is this a personal distress message?
   (Must be a first-person account of an individual's struggle).

Respond in the following format:
THOUGHTS: [Brief reasoning]
DISTRESS: [YES/NO]
CULTURE: [YES/NO]

Text: "{text}"
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=50
    )

    result = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    distress = "distress: yes" in result
    culture = "culture: yes" in result

    return distress and culture

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import pandas as pd

df_spanish = pd.read_csv("/content/drive/MyDrive/mental_health/reddit_spanish_raw.csv")

In [ ]:
filtered_rows = []

batch_size = 10

for i in range(0, len(df_spanish), batch_size):

    batch = df_spanish.iloc[i:i+batch_size]

    print(f"Processing batch {i} - {min(i+batch_size, len(df_spanish))}")

    for _, row in batch.iterrows():

        text = row["text"]

        try:
            if not isinstance(text, str) or len(text) < 20:
                continue

            if llm_filter_spanish(text):
                filtered_rows.append(row)

        except Exception as e:
            print("error:", e)

    if i % 5 == 0:
        temp_df_spanish = pd.DataFrame(filtered_rows)
        temp_df_spanish.to_csv("/content/drive/MyDrive/mental_health/backup_filtered_spanish.csv", index=False)


df_spanish_filtered = pd.DataFrame(filtered_rows).reset_index(drop=True)

df_spanish_filtered.to_csv(
    "/content/drive/MyDrive/mental_health/filtered_llm_spanish.csv",
    index=False
)

Processing batch 0 - 10
Processing batch 10 - 20
Processing batch 20 - 30
Processing batch 30 - 40
Processing batch 40 - 50
Processing batch 50 - 60
Processing batch 60 - 70
Processing batch 70 - 80
Processing batch 80 - 90
Processing batch 90 - 100
Processing batch 100 - 110
Processing batch 110 - 120
Processing batch 120 - 130
Processing batch 130 - 140
Processing batch 140 - 150
Processing batch 150 - 160
Processing batch 160 - 170
Processing batch 170 - 180
Processing batch 180 - 190
Processing batch 190 - 200
Processing batch 200 - 210
Processing batch 210 - 220
Processing batch 220 - 230
Processing batch 230 - 240
Processing batch 240 - 250
Processing batch 250 - 260
Processing batch 260 - 270
Processing batch 270 - 280
Processing batch 280 - 290
Processing batch 290 - 300
Processing batch 300 - 310
Processing batch 310 - 320
Processing batch 320 - 330
Processing batch 330 - 340
Processing batch 340 - 350
Processing batch 350 - 360
Processing batch 360 - 370
Processing batch 370 